# Animal Slaughter GP Dashboard

Five interactive sections:
1. **GP Posterior Plot** — interactive country / species / year selector
2. **UMAP of GP Parameters** — trajectory shape embedding, coloured by species or country
3. **Dimensionality Reduction of Raw Data** — UMAP, PCA scatter, PC1 histogram, T-SNE
4. **Ranked Gradients** — annual fractional growth rate per (country, species)
5. **Ranked Relative Gradients** — growth rate normalised by current slaughter count

In [14]:
import sys
!{sys.executable} -m pip install plotly ipywidgets umap-learn jupyterlab-widgets widgetsnbextension nbformat -q

> **Restart the kernel** after the install cell if you have not done so yet, then run all cells.

In [15]:
import pickle
import warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import umap

warnings.filterwarnings('ignore')

GP_EXTENDED_PATH = 'gp_results_extended.pkl'
GP_BASE_PATH     = 'gp_results.pkl'
CSV_PATH         = 'animals-slaughtered-for-meat/animals-slaughtered-for-meat.csv'
SPECIES          = ['Cattle', 'Goat', 'Chicken', 'Turkey', 'Pig', 'Sheep', 'Duck']
PRED_YEARS       = np.linspace(1961, 2032, 300)
IDX_2022         = int(np.argmin(np.abs(PRED_YEARS - 2022)))

SPECIES_COLORS = {
    'Cattle':  '#1f77b4',
    'Goat':    '#ff7f0e',
    'Chicken': '#2ca02c',
    'Turkey':  '#d62728',
    'Pig':     '#9467bd',
    'Sheep':   '#8c564b',
    'Duck':    '#e377c2',
}

def hex_to_rgba(hex_color, alpha):
    h = hex_color.lstrip('#')
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return f'rgba({r},{g},{b},{alpha})'

In [16]:
with open(GP_EXTENDED_PATH, 'rb') as f:
    gp_ext = pickle.load(f)
with open(GP_BASE_PATH, 'rb') as f:
    gp_base = pickle.load(f)
df_raw = pd.read_csv(CSV_PATH)

ext_countries  = sorted(gp_ext.keys())
base_countries = sorted(gp_base.keys())

print(f'Extended pkl : {len(ext_countries)} countries x 7 species')
print(f'Base pkl     : {len(base_countries)} countries x Cattle only')
print(f'Raw CSV      : {len(df_raw)} rows, years {df_raw["Year"].min()}–{df_raw["Year"].max()}')

Extended pkl : 25 countries x 7 species
Base pkl     : 251 countries x Cattle only
Raw CSV      : 14690 rows, years 1961–2024


---
## Section 1 — Interactive GP Posterior Plot

Select species, countries and a year range. The ribbon is ±1 posterior standard deviation (actual scale, not log).

- **25 countries** have all 7 species (from `gp_results_extended.pkl`)
- **251 countries** are available for **Cattle only** (from `gp_results.pkl`) — select Cattle only to expand the country list.

In [17]:
def build_gp_traces(selected_countries, selected_species):
    traces = []
    for country in selected_countries:
        for sp in selected_species:
            entry = gp_ext.get(country, {}).get(sp)
            if entry is None and sp == 'Cattle':
                entry = gp_base.get(country, {}).get('Cattle')
            if entry is None:
                continue
            color = SPECIES_COLORS[sp]
            x = entry['years_pred']
            mean_act  = np.exp(entry['mean'])
            upper_act = np.exp(entry['mean'] + entry['std'])
            lower_act = np.exp(entry['mean'] - entry['std'])
            label = f'{country} – {sp}'
            traces.append(go.Scatter(
                x=x, y=upper_act, mode='lines',
                line=dict(width=0), showlegend=False,
                hoverinfo='skip', name=label + ' upper'
            ))
            traces.append(go.Scatter(
                x=x, y=lower_act, mode='lines', fill='tonexty',
                fillcolor=hex_to_rgba(color, 0.15),
                line=dict(width=0), showlegend=False,
                hoverinfo='skip', name=label + ' lower'
            ))
            traces.append(go.Scatter(
                x=x, y=mean_act, mode='lines',
                line=dict(color=color, width=2), name=label
            ))
            traces.append(go.Scatter(
                x=entry['obs_years'], y=np.exp(entry['obs_log_values']),
                mode='markers', marker=dict(color=color, size=5),
                showlegend=False, name=label + ' obs',
                hovertemplate='%{x:.0f}: %{y:,.0f}<extra>' + label + '</extra>'
            ))
    return traces

In [28]:
year_slider = widgets.IntRangeSlider(
    value=[1990, 2032], min=1961, max=2032, step=1,
    description='Years:', continuous_update=False,
    layout=widgets.Layout(width='600px')
)
species_sel = widgets.SelectMultiple(
    options=SPECIES, value=['Cattle'], description='Species:',
    layout=widgets.Layout(height='148px', width='150px')
)
country_sel = widgets.SelectMultiple(
    options=ext_countries, value=ext_countries[:3], description='Country:',
    layout=widgets.Layout(height='200px', width='230px')
)
info_label = widgets.HTML(
    '<small><i>25 countries for all species. '
    'Select Cattle only to access 251 countries.</i></small>'
)
scale_toggle = widgets.ToggleButtons(
    options=['Log scale', 'Linear scale'], value='Log scale',
    layout=widgets.Layout(width='260px')
)
out1 = widgets.Output()

def _update_plot(change=None):
    sel_sp = list(species_sel.value)
    sel_co = list(country_sel.value)
    if sel_sp == ['Cattle']:
        if list(country_sel.options) != base_countries:
            country_sel.options = base_countries
    else:
        if list(country_sel.options) != ext_countries:
            country_sel.options = ext_countries
            country_sel.value = [c for c in sel_co if c in ext_countries] or ext_countries[:1]
    traces = build_gp_traces(list(country_sel.value), sel_sp)
    yr_lo, yr_hi = year_slider.value
    ytype = 'log' if scale_toggle.value == 'Log scale' else 'linear'
    fig = go.Figure(traces)
    fig.update_layout(
        title='Animals slaughtered — GP posterior',
        xaxis=dict(title='Year', range=[yr_lo, yr_hi]),
        yaxis=dict(title='Animals slaughtered (head)', type=ytype),
        hovermode='x unified', height=520,
    )
    out1.clear_output(wait=True)
    with out1:
        fig.show()

year_slider.observe(_update_plot, names='value')
species_sel.observe(_update_plot, names='value')
country_sel.observe(_update_plot, names='value')
scale_toggle.observe(_update_plot, names='value')
_update_plot()

display(widgets.VBox([
    widgets.HBox([species_sel, country_sel, info_label]),
    widgets.HBox([year_slider, scale_toggle]),
    out1
]))

---
## Section 2 — UMAP of GP Parameters

Each point is a (country, species) pair. The feature vector is the GP posterior mean sampled at 20 evenly-spaced years — it encodes the *shape* of the trajectory (log scale). Use the dropdown to colour by species or country.

In [19]:
SAMPLE_IDX = np.linspace(0, 299, 20, dtype=int)

rows_gp, ctry_gp, sp_gp = [], [], []
for country in ext_countries:
    for sp in SPECIES:
        entry = gp_ext[country].get(sp)
        if entry is None:
            continue
        rows_gp.append(entry['mean'][SAMPLE_IDX])
        ctry_gp.append(country)
        sp_gp.append(sp)

X_gp = np.array(rows_gp)
df_gp = pd.DataFrame({'country': ctry_gp, 'species': sp_gp})
print(f'GP feature matrix: {X_gp.shape}')

GP feature matrix: (171, 20)


In [20]:
reducer_gp = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=42)
X_gp_2d = reducer_gp.fit_transform(X_gp)
df_gp['umap_x'] = X_gp_2d[:, 0]
df_gp['umap_y'] = X_gp_2d[:, 1]
print('UMAP done')

UMAP done


In [ ]:
SPECIES_SYMBOLS = {
    'Cattle':  'circle',
    'Goat':    'triangle-up',
    'Chicken': 'diamond',
    'Turkey':  'square',
    'Pig':     'pentagon',
    'Sheep':   'star',
    'Duck':    'hexagram',
}

CONTINENT_MAP = {
    'Australia': 'Oceania',    'New Zealand': 'Oceania',
    'Argentina': 'S. America', 'Brazil': 'S. America',
    'France': 'Europe',   'Germany': 'Europe',  'Netherlands': 'Europe',
    'Spain': 'Europe',    'Italy': 'Europe',    'Poland': 'Europe',
    'Hungary': 'Europe',  'Norway': 'Europe',   'Greece': 'Europe',
    'Portugal': 'Europe', 'Austria': 'Europe',  'Sweden': 'Europe',
    'Japan': 'Asia',      'Iran': 'Asia',       'India': 'Asia',
    'China': 'Asia',      'Philippines': 'Asia',
    'United States': 'N. America', 'Mexico': 'N. America',
    'South Africa': 'Africa',      'Egypt': 'Africa',
}

CONTINENT_COLORS = {
    'Europe':     '#1f77b4',
    'Asia':       '#ff7f0e',
    'N. America': '#2ca02c',
    'S. America': '#d62728',
    'Africa':     '#9467bd',
    'Oceania':    '#8c564b',
}

df_gp['continent'] = df_gp['country'].map(CONTINENT_MAP)

traces = []

# One trace per continent — mixed symbols per point (array passed to marker.symbol)
for i, (cont, color) in enumerate(CONTINENT_COLORS.items()):
    sub = df_gp[df_gp['continent'] == cont]
    if sub.empty:
        continue
    traces.append(go.Scatter(
        x=sub['umap_x'], y=sub['umap_y'],
        mode='markers',
        marker=dict(
            symbol=[SPECIES_SYMBOLS[sp] for sp in sub['species']],
            color=color, size=13,
            line=dict(width=1, color='white'),
        ),
        name=cont,
        legendgroup='continent',
        legendgrouptitle_text='Continent' if i == 0 else None,
        hovertext=sub['country'] + ' – ' + sub['species'],
        hovertemplate='%{hovertext}<extra></extra>',
    ))

# Invisible dummy traces just to show the species symbol legend
for j, (sp, symbol) in enumerate(SPECIES_SYMBOLS.items()):
    traces.append(go.Scatter(
        x=[None], y=[None],
        mode='markers',
        marker=dict(symbol=symbol, color='gray', size=10),
        name=sp,
        legendgroup='species',
        legendgrouptitle_text='Species' if j == 0 else None,
    ))

fig2 = go.Figure(data=traces)
fig2.update_layout(
    title='UMAP of GP Posterior Trajectories',
    xaxis_title='UMAP 1', yaxis_title='UMAP 2', height=600,
    legend=dict(groupclick='toggleitem'),
)

out2 = widgets.Output()
with out2:
    fig2.show()
display(out2)

---
## Section 3 — Dimensionality Reduction of Raw Slaughter Data

Feature matrix: one row per (country, species), columns = log-slaughter at each year 1961–2022.
Rows with fewer than 10 valid observations or >50% NaN are dropped; remaining NaNs are median-imputed.

> **Note:** T-SNE may take ~60 seconds.

In [22]:
YEARS_RAW   = list(range(1961, 2023))
NAN_THRESH  = 0.5
MIN_OBS_RAW = 10

records3, ctry3, sp3 = [], [], []
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    for sp in SPECIES:
        for country in sorted(df_raw['Entity'].unique()):
            sub = (df_raw[df_raw['Entity'] == country][['Year', sp]]
                   .set_index('Year').reindex(YEARS_RAW))
            vals = sub[sp].values.astype(float)
            valid = (vals > 0) & ~np.isnan(vals)
            if valid.sum() < MIN_OBS_RAW:
                continue
            log_vals = np.where(valid, np.log(vals), np.nan)
            if np.isnan(log_vals).mean() > NAN_THRESH:
                continue
            records3.append(log_vals)
            ctry3.append(country)
            sp3.append(sp)

X3_raw = np.array(records3)
imp3    = SimpleImputer(strategy='median')
X3_imp  = imp3.fit_transform(X3_raw)
scaler3 = StandardScaler()
X3      = scaler3.fit_transform(X3_imp)
df3     = pd.DataFrame({'country': ctry3, 'species': sp3})
print(f'Raw feature matrix: {X3.shape}, NaN after impute: {np.isnan(X3).sum()}')

Raw feature matrix: (1325, 62), NaN after impute: 0


In [23]:
pca3    = PCA(n_components=2, random_state=42)
X3_pca  = pca3.fit_transform(X3)
ev1     = pca3.explained_variance_ratio_[0] * 100
ev2     = pca3.explained_variance_ratio_[1] * 100
print(f'PCA variance: PC1={ev1:.1f}%  PC2={ev2:.1f}%')

reducer3 = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=42)
X3_umap  = reducer3.fit_transform(X3)
print('UMAP done')

df3['pca_x']  = X3_pca[:, 0]
df3['pca_y']  = X3_pca[:, 1]
df3['umap_x'] = X3_umap[:, 0]
df3['umap_y'] = X3_umap[:, 1]

PCA variance: PC1=94.8%  PC2=3.6%
UMAP done


In [24]:
# ⚠️ T-SNE can take ~60 seconds
tsne3   = TSNE(n_components=2, perplexity=30, max_iter=1000,
               init='pca', random_state=42)
X3_tsne = tsne3.fit_transform(X3)
df3['tsne_x'] = X3_tsne[:, 0]
df3['tsne_y'] = X3_tsne[:, 1]
print('T-SNE done')

T-SNE done


In [25]:
all_countries3 = sorted(df3['country'].unique())
country_colors3 = {c: country_palette[i % len(country_palette)]
                   for i, c in enumerate(all_countries3)}

color_by3 = widgets.Dropdown(
    options=['Species', 'Country'], value='Species',
    description='Color by:', layout=widgets.Layout(width='200px')
)
out3 = widgets.Output()

def _make_fig3(color_by):
    color_col = 'species' if color_by == 'Species' else 'country'
    if color_by == 'Species':
        color_map = SPECIES_COLORS
        cat_order = SPECIES
    else:
        color_map = country_colors3
        cat_order = all_countries3

    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=[
            'UMAP',
            f'PCA  (PC1={ev1:.1f}%, PC2={ev2:.1f}%)',
            'PC1 score histogram',
            'T-SNE'
        ]
    )
    shown = set()
    for cat in cat_order:
        sub = df3[df3[color_col] == cat]
        if sub.empty:
            continue
        color = color_map.get(cat, '#888888')
        show_legend = cat not in shown
        shown.add(cat)
        kw = dict(
            mode='markers',
            marker=dict(color=color, size=5, opacity=0.7),
            name=cat, legendgroup=cat, showlegend=show_legend,
            text=sub['country'] + ' – ' + sub['species'],
            hovertemplate='%{text}<extra></extra>'
        )
        fig.add_trace(go.Scatter(x=sub['umap_x'], y=sub['umap_y'], **kw), row=1, col=1)
        fig.add_trace(go.Scatter(x=sub['pca_x'],  y=sub['pca_y'],  **kw), row=1, col=2)
        fig.add_trace(go.Scatter(x=sub['tsne_x'], y=sub['tsne_y'], **kw), row=2, col=2)
        fig.add_trace(go.Histogram(
            x=sub['pca_x'], name=cat, legendgroup=cat, showlegend=False,
            marker_color=color, opacity=0.65,
            hovertemplate=cat + '<br>PC1=%{x:.2f}, count=%{y}<extra></extra>'
        ), row=2, col=1)

    fig.update_xaxes(title_text='UMAP 1',  row=1, col=1)
    fig.update_yaxes(title_text='UMAP 2',  row=1, col=1)
    fig.update_xaxes(title_text='PC1',     row=1, col=2)
    fig.update_yaxes(title_text='PC2',     row=1, col=2)
    fig.update_xaxes(title_text='PC1',     row=2, col=1)
    fig.update_yaxes(title_text='Count',   row=2, col=1)
    fig.update_xaxes(title_text='T-SNE 1', row=2, col=2)
    fig.update_yaxes(title_text='T-SNE 2', row=2, col=2)
    fig.update_layout(
        height=840, barmode='overlay',
        title_text='Dimensionality Reduction of Raw Slaughter Data'
    )
    return fig

def _update_fig3(change=None):
    out3.clear_output(wait=True)
    with out3:
        _make_fig3(color_by3.value).show()

color_by3.observe(_update_fig3, names='value')
_update_fig3()
display(widgets.VBox([color_by3, out3]))

---
## Section 4 — Ranked Gradient List

`gradient_now` = rate of change of log-slaughter at 2022 (log-units / year ≈ annual fractional growth rate). Positive = increasing, negative = decreasing.

Source: `gp_results_extended.pkl` (25 countries × 7 species).

In [26]:
grad_rows = []
for country in ext_countries:
    for sp in SPECIES:
        entry = gp_ext[country].get(sp)
        if entry is None:
            continue
        grad_rows.append({
            'country': country, 'species': sp,
            'gradient_now': float(entry['gradient_now']),
            'label': f'{country} ({sp})'
        })
df_grad = pd.DataFrame(grad_rows)
print(f'{len(df_grad)} valid fits')

sp_filter4 = widgets.SelectMultiple(
    options=['All'] + SPECIES, value=['All'],
    description='Species:', layout=widgets.Layout(height='185px', width='150px')
)
out4 = widgets.Output()

def _update_fig4(change=None):
    sel = list(sp_filter4.value)
    df_f = df_grad if 'All' in sel else df_grad[df_grad['species'].isin(sel)]
    df_f = df_f.sort_values('gradient_now', ascending=True)
    fig = go.Figure(go.Bar(
        x=df_f['gradient_now'], y=df_f['label'], orientation='h',
        marker_color=[SPECIES_COLORS[s] for s in df_f['species']],
        hovertemplate='%{y}<br>Gradient: %{x:.4f} log-units/yr<extra></extra>'
    ))
    fig.add_vline(x=0, line_dash='dash', line_color='gray', line_width=1)
    fig.update_layout(
        title='Ranked Annual Gradient (≈ fractional growth rate)',
        xaxis_title='gradient_now  (log-units / year)',
        height=max(400, 22 * len(df_f)),
        margin=dict(l=220), showlegend=False
    )
    out4.clear_output(wait=True)
    with out4:
        fig.show()

sp_filter4.observe(_update_fig4, names='value')
_update_fig4()
display(widgets.HBox([sp_filter4, out4]))

171 valid fits


---
## Section 5 — Ranked Relative Gradient List

`relative_gradient = gradient_now / actual_2022`, where `actual_2022 = exp(GP mean at 2022)`.

This normalises by scale: a country with 10 M slaughters growing at the same log-rate as one with 100 K has the same gradient, but a much smaller *relative* gradient. Countries where absolute growth is small but the herd is tiny will rank highly here.

Hover to see the actual 2022 count and raw gradient.

In [27]:
relgrad_rows = []
for country in ext_countries:
    for sp in SPECIES:
        entry = gp_ext[country].get(sp)
        if entry is None:
            continue
        gradient_now = float(entry['gradient_now'])
        actual_2022  = float(np.exp(entry['mean'][IDX_2022]))
        rel_grad     = gradient_now / actual_2022
        relgrad_rows.append({
            'country': country, 'species': sp,
            'gradient_now': gradient_now,
            'actual_2022':  actual_2022,
            'rel_grad':     rel_grad,
            'label': f'{country} ({sp})'
        })
df_relgrad = pd.DataFrame(relgrad_rows)

sp_filter5 = widgets.SelectMultiple(
    options=['All'] + SPECIES, value=['All'],
    description='Species:', layout=widgets.Layout(height='185px', width='150px')
)
out5 = widgets.Output()

def _update_fig5(change=None):
    sel = list(sp_filter5.value)
    df_f = df_relgrad if 'All' in sel else df_relgrad[df_relgrad['species'].isin(sel)]
    df_f = df_f.sort_values('rel_grad', ascending=True)
    fig = go.Figure(go.Bar(
        x=df_f['rel_grad'], y=df_f['label'], orientation='h',
        marker_color=[SPECIES_COLORS[s] for s in df_f['species']],
        customdata=np.stack([df_f['actual_2022'], df_f['gradient_now']], axis=1),
        hovertemplate=(
            '%{y}<br>'
            'Rel. gradient: %{x:.3e} yr\u207b\u00b9 per animal<br>'
            'Actual 2022: %{customdata[0]:,.0f}<br>'
            'Gradient: %{customdata[1]:.4f}<extra></extra>'
        )
    ))
    fig.add_vline(x=0, line_dash='dash', line_color='gray', line_width=1)
    fig.update_layout(
        title='Ranked Relative Gradient  (gradient_now / actual 2022 count)',
        xaxis_title='gradient_now / actual_2022  (yr\u207b\u00b9 per animal)',
        height=max(400, 22 * len(df_f)),
        margin=dict(l=220), showlegend=False
    )
    out5.clear_output(wait=True)
    with out5:
        fig.show()

sp_filter5.observe(_update_fig5, names='value')
_update_fig5()
display(widgets.HBox([sp_filter5, out5]))